# Finding similar items
The aim is to detect similar textual items in the `text` field of the Kaggle [Yelp](https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset) dataset.

In [11]:
import os
import json
import pandas as pd
import pip
import string

def import_or_install(package):
    try:
        __import__(package)
    except ImportError:
        pip.main(['install', package])



In [12]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark
findspark.init("spark-3.5.0-bin-hadoop3")# SPARK_HOME
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

We first of all import the Yelp dataset from Kaggle, using a token.

In [13]:
os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"
!kaggle datasets download -d yelp-dataset/yelp-dataset

yelp-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [14]:
from tqdm import tqdm
import zipfile

DATA_DIR = "/content/yelp-dataset"

with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
     for file in tqdm(iterable=zip_ref.namelist(), total=len(zip_ref.namelist())):
          zip_ref.extract(member=file, path=DATA_DIR)

100%|██████████| 6/6 [01:31<00:00, 15.33s/it]


We take into consideration the portion of the dataset regarding reviews, contained in `yelp_academic_dataset_review.json`. We convert the resulting dataframe in RDD form.

In [15]:
df_reviews = spark.read.json(DATA_DIR + "/yelp_academic_dataset_review.json")
reviews_RDD = df_reviews.rdd

Seen the aim of the project we will only be looking at the `text` attribute of the imported reviews.

In [49]:
text_RDD = reviews_RDD.map((lambda r: (r[0], r['text'])))

Let's look at some reviews of the text field of the review dataset.

In [50]:
print('---\n')
for text in text_RDD.take(4):
    print('{}\n'.format(text[1]))
    print('---\n')

---

If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. 

The food is good, but it takes a very long time to come out. The waitstaff is very young, but usually pleasant. We have just had too many experiences where we spent way too long waiting. We usually opt for another diner or restaurant on the weekends, in order to be done quicker.

---

I've taken a lot of spin classes over the years, and nothing compares to the classes at Body Cycle. From the nice, clean space and amazing bikes, to the welcoming and motivating instructors, every class is a top notch work out.

For anyone who struggles to fit workouts in, the online scheduling system makes it easy to plan ahead (and there's no need to line up way in advanced like many gyms make you do).

There is no way I can write this review without giving Russell, th

## Data pre-processing

We begin by removing text cells that are `None` or that contain empty strings. It is possible to verify that for each review a nonempty text is present.

In [51]:
text_RDD = text_RDD.filter(lambda text: bool(text))

To study the similarity between the texts of the reviews we proceed by looking at the relative string as a set of tokens. In other words we divide each review in the terms, all considered in lower case, that compose it.

We get rid of the stop words appearing in the tokens to extract the actual semantics of the text. Also we lemmatize the remaining tokens to consider as similar the various inflections of a certain word.

In [62]:
%%capture
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stopwords = stopwords.words('english')
import spacy
nlp = spacy.load("en_core_web_sm")

split_regex = r'\W+'

tokenize = lambda string: [s for s in re.split(split_regex,string.lower()) if s not in stopwords and s != '']
text_tokens_RDD = text_RDD.map(lambda s: (s[0],tokenize(s[1])))

lemmatize = lambda word: str([token.lemma_ for token in nlp(word)][0])
text_lemmatized_tokens_RDD = text_tokens_RDD.map(lambda s: (s[0], [lemmatize(w) for w in s[1]]))

Let's look at how the first of the reviews has been transformed.

In [63]:
text_lemmatized_tokens_RDD.first()

('XQfwVwDr-v0ZS3_CbbE5Xw',
 ['decide',
  'eat',
  'aware',
  'go',
  'take',
  '2',
  'hour',
  'begin',
  'end',
  'try',
  'multiple',
  'time',
  'want',
  'like',
  'location',
  'nj',
  'never',
  'bad',
  'experience',
  'food',
  'good',
  'take',
  'long',
  'time',
  'come',
  'waitstaff',
  'young',
  'usually',
  'pleasant',
  'many',
  'experience',
  'spend',
  'way',
  'long',
  'wait',
  'usually',
  'opt',
  'another',
  'diner',
  'restaurant',
  'weekend',
  'order',
  'do',
  'quicker'])

At this point we have codified the text of each review through it's essential information.